# TrustShift-IDS WP2 — V10: Paired Comparison, State-Transfer Audit, and Source-Only Ablations

**Purpose.** Analyze the saved V9 results without retraining, verify whether a historical XGBoost state sequence can be aligned to the same `(seed, window)` deployment units, and run label-free three-family ablations. The notebook **fails closed** for unverified transferred-state comparisons.

**Important limitations.**
- Historical XGBoost states cannot be substituted with clean-IID regression states (82 windows) or assigned by row order.
- The existing V9 per-window policy CSV contains only four policy outcomes. Exact counterfactual outcomes for arbitrary transferred or ablated state sequences require the saved per-flow probabilities and labels, or a new inference pass. This notebook never fabricates those outcomes.
- Four seeds are not four independent datasets. Block-bootstrap uncertainty is descriptive, not a formal guarantee.
- V9's `bcdr` depends on the window-level 1% criterion; treat it as a study-specific operational metric.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, re, warnings
import numpy as np
import pandas as pd
from scipy import stats
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/TrustShift_IDS')
V2=ROOT/'V2_adaptive_reliability'
OUT=V2/'WP2_detector_specific'
assert OUT.is_dir(), f'Missing output directory: {OUT}'
FILES={
    'monitor':OUT/'WP2_V9_label_free_monitor_windows.csv',
    'policy':OUT/'WP2_V9_window_policy_results.csv',
    'summary':OUT/'WP2_V9_policy_summary.csv',
    'mlp_null':OUT/'WP2_V9_MLP_source_null_raw.csv',
    'ft_null':OUT/'WP2_V9_FTTransformer_source_null_raw.csv',
    'mlp_cfg':OUT/'WP2_V9_MLP_frozen_monitor.json',
    'ft_cfg':OUT/'WP2_V9_FTTransformer_frozen_monitor.json',
}
for k,p in FILES.items():
    assert p.exists(), f'{k} not found: {p}'
monitor=pd.read_csv(FILES['monitor'])
results=pd.read_csv(FILES['policy'])
print('V9 files verified',len(monitor),'monitor rows;',len(results),'policy rows')
assert len(monitor)==480 and len(results)==1920
assert not monitor.duplicated(['seed','window','detector']).any()
assert not results.duplicated(['seed','window','detector','policy']).any()
assert set(monitor.seed)=={1,7,21,42} and set(monitor.window)==set(range(60))
assert set(results.policy)=={'no_abstention','fixed_normal','static_severe','detector_specific_3family'}
assert set(monitor.state)=={'normal','elevated','severe'}
print('V10 input integrity: PASS')


## C1 — Descriptive and paired comparisons from saved V9 outcomes

For a policy \(p\), accepted risk is pooled accepted errors / pooled accepted samples, while coverage is pooled accepted samples / all samples. We report per-window budget violations separately. Comparisons are paired on detector, seed, and window.


In [ ]:
EPS=.01
def aggregate(g):
    a=g.acc_n.sum(); n=g.n.sum(); attacks=g.n_attack.sum()
    risk=np.where(g.acc_n>0,g.acc_err/g.acc_n,np.nan)
    compliant=np.isfinite(risk)&(risk<=EPS)
    return pd.Series(dict(windows=len(g),coverage=a/n,accepted_risk=g.acc_err.sum()/max(a,1),
        violations=int(np.sum(risk>EPS)),attack_deferral=g.deferred_attack.sum()/max(attacks,1),
        auto_attack_recall=g.acc_detected_attack.sum()/max(attacks,1),
        false_alarms=int(g.acc_false_alarm.sum()),
        bcdr=g.loc[compliant,'acc_detected_attack'].sum()/max(attacks,1)))
summary=results.groupby(['detector','policy']).apply(aggregate,include_groups=False).reset_index()
summary.to_csv(OUT/'WP2_V10_verified_V9_summary.csv',index=False)
display(summary)
# Paired differences at the identical window granularity
w=results.copy()
w['window_risk']=np.where(w.acc_n>0,w.acc_err/w.acc_n,np.nan)
w['window_coverage']=w.acc_n/w.n
wide=w.pivot(index=['detector','seed','window'],columns='policy',values=['window_risk','window_coverage'])
pairs=[]
for det in sorted(results.detector.unique()):
    for comparator in ['fixed_normal','static_severe','no_abstention']:
        sub=wide.loc[det]
        for metric in ['window_risk','window_coverage']:
            delta=sub[(metric,'detector_specific_3family')]-sub[(metric,comparator)]
            valid=delta.dropna()
            pairs.append(dict(detector=det,comparator=comparator,metric=metric,
                              paired_windows=len(valid),mean_window_difference=valid.mean(),
                              median_window_difference=valid.median(),
                              positive_fraction=(valid>0).mean()))
paired=pd.DataFrame(pairs)
paired.to_csv(OUT/'WP2_V10_paired_window_descriptives.csv',index=False)
display(paired)


## C2 — Dependence-aware descriptive bootstrap

Resample contiguous **10-window blocks within each seed**, retaining both policy outcomes for each sampled block. This captures local temporal dependence better than an IID window bootstrap, but **does not** account for cross-seed dependence arising from overlapping underlying flows. Confidence intervals should therefore be described as *exploratory block-bootstrap intervals*, not inferential proof of generalization.


In [ ]:
RNG=np.random.default_rng(20261104)
B=1500
BLOCK=10
comparisons=[]
for det in sorted(results.detector.unique()):
    sub=results[results.detector==det]
    for comp in ['fixed_normal','static_severe','no_abstention']:
        diffs=[]
        for _ in range(B):
            picks=[]
            for seed in [1,7,21,42]:
                starts=RNG.integers(0,60-BLOCK+1,size=6)
                for st in starts:
                    picks.extend((seed,int(x)) for x in range(st,st+BLOCK))
            # multiplicity-preserving merge
            pick_df=pd.DataFrame(picks,columns=['seed','window'])
            sample=pick_df.merge(sub,on=['seed','window'],how='left',validate='many_to_many')
            agg=sample.groupby('policy').apply(aggregate,include_groups=False)
            diffs.append([agg.loc['detector_specific_3family','accepted_risk']-agg.loc[comp,'accepted_risk'],
                          agg.loc['detector_specific_3family','coverage']-agg.loc[comp,'coverage']])
        a=np.asarray(diffs)
        for j,metric in enumerate(['accepted_risk','coverage']):
            comparisons.append(dict(detector=det,comparator=comp,metric=metric,
                                    delta_point=float(summary.set_index(['detector','policy']).loc[(det,'detector_specific_3family'),metric]-
                                                      summary.set_index(['detector','policy']).loc[(det,comp),metric]),
                                    bootstrap_q025=float(np.quantile(a[:,j],.025)),
                                    bootstrap_q975=float(np.quantile(a[:,j],.975)),
                                    bootstrap_type='exploratory within-seed moving-block'))
boot=pd.DataFrame(comparisons)
boot.to_csv(OUT/'WP2_V10_exploratory_block_bootstrap.csv',index=False)
display(boot)


## C3 — Source-only monitoring-family ablations

Ablation removes one entire family from the SSS and **recalibrates the q95/q99 boundaries using only the 54 saved source-null windows**. Deployment labels are not accessed for this step.

**Caution:** With only two families, the median is their arithmetic midpoint. Thus this is a sensitivity analysis, not an exact recreation of the original three-family rule. The ablation scores are not evidence of improved detection unless paired outcome evaluation is performed later.


In [ ]:
IND_CONF=['binary_confidence_shift','binary_entropy_shift','binary_margin_shift',
          'multiclass_confidence_shift','multiclass_entropy_shift','multiclass_margin_shift']
IND_CLASS=['binary_prediction_js','multiclass_prediction_js']
IND_FEATURE=['feature_shift_mean','feature_shift_median','feature_shift_p90','feature_shift_max']
FAMILIES={'confidence':IND_CONF,'predicted_class':IND_CLASS,'feature':IND_FEATURE}
configs={}
for det in ['MLP','FTTransformer']:
    with open(FILES['mlp_cfg' if det=='MLP' else 'ft_cfg']) as f:configs[det]=json.load(f)
def calc_score(row,norm,active,raw_prefix=''):
    fam=[]
    for family in active:
        e=[]
        for indicator in FAMILIES[family]:
            c=norm[indicator]
            v=float(row[raw_prefix+indicator])
            if c['structural_zero']:val=float(v>0)
            elif c['scale']<=0:val=0.
            else:val=max(0.,(v-c['median'])/c['scale'])
            e.append(val)
        fam.append(np.median(e))
    return float(np.median(fam))
ablation=[]
for det in ['MLP','FTTransformer']:
    null=pd.read_csv(FILES['mlp_null' if det=='MLP' else 'ft_null'])
    norm=configs[det]['normalization']
    m=monitor[monitor.detector==det].copy()
    for removed in [None,*FAMILIES.keys()]:
        active=[f for f in FAMILIES if f!=removed]
        source_scores=np.array([calc_score(row,norm,active) for row in null.to_dict('records')])
        lo,hi=np.quantile(source_scores,[.95,.99])
        for row in m.to_dict('records'):
            s=calc_score(row,norm,active,raw_prefix='raw_')
            state='normal' if s<=lo else 'elevated' if s<=hi else 'severe'
            ablation.append(dict(detector=det,seed=int(row['seed']),window=int(row['window']),
                                 removed_family=removed or 'none',SSS=s,state=state,
                                 q95=lo,q99=hi))
ab=pd.DataFrame(ablation)
ab.to_csv(OUT/'WP2_V10_source_only_family_ablation_states.csv',index=False)
display(ab.groupby(['detector','removed_family','state']).size().unstack(fill_value=0))
# The all-family recomputation must exactly reproduce V9, otherwise stop.
full=ab[ab.removed_family=='none'].merge(monitor[['detector','seed','window','state','SSS']],
                  on=['detector','seed','window'],suffixes=('_recomputed','_V9'),validate='one_to_one')
assert (full.state_recomputed==full.state_V9).all(), 'V9 state regression failed'
assert np.allclose(full.SSS_recomputed,full.SSS_V9,rtol=0,atol=1e-9),'V9 SSS regression failed'
print('C3 V9 full-family state and score regression: PASS')


## C4 — Find candidate transferred-XGBoost monitoring states (strict alignment)

This inventory looks for saved CSV/Parquet monitoring records. It **does not** infer seed/window alignment from row count, filenames, or historical 82-window clean-IID states. Only files explicitly containing `(seed, window)` and a state column can pass the initial schema check.

**If no aligned historical state file is found, V10 reports that comparison as blocked.** Do not interpret an absent baseline as a negative result.


In [ ]:
import os
candidates=[]
for base in [V2,ROOT/'results']:
    if not base.exists():continue
    for dirpath,_,files in os.walk(base):
        # skip expensive model dirs and WP2 V9/V10 outputs
        if '/WP2_detector_specific' in dirpath:continue
        for filename in files:
            if not filename.lower().endswith(('.csv','.parquet')):continue
            if not any(k in filename.lower() for k in ['state','monitor','stream','shift','seed','sss']):continue
            path=Path(dirpath)/filename
            candidates.append(path)
print('Potential source files:',len(candidates))
inventory=[]
valid=[]
for path in candidates:
    try:
        df=pd.read_csv(path,nrows=6) if path.suffix.lower()=='.csv' else pd.read_parquet(path)
        cols=list(df.columns)
        state_cols=[c for c in cols if 'state' in c.lower()]
        seed_cols=[c for c in cols if c.lower() in ('seed','random_seed','ordering_seed')]
        window_cols=[c for c in cols if c.lower() in ('window','window_id','window_index')]
        eligible=bool(state_cols and seed_cols and window_cols)
        inventory.append(dict(path=str(path),rows_preview=len(df),state_columns=';'.join(state_cols),
                              seed_columns=';'.join(seed_cols),window_columns=';'.join(window_cols),
                              schema_candidate=eligible))
        if eligible:valid.append((path,seed_cols[0],window_cols[0],state_cols[0]))
    except Exception as exc:
        inventory.append(dict(path=str(path),error=str(exc)[:150],schema_candidate=False))
inv=pd.DataFrame(inventory)
inv.to_csv(OUT/'WP2_V10_xgboost_state_candidate_inventory.csv',index=False)
print('Schema-compatible candidates:',len(valid))
if len(valid):display(inv[inv.schema_candidate].head(30))
else:print('TRANSFERRED-STATE BASELINE: BLOCKED — no verified seed/window state file found')


## C5 — Verify alignment, never guess it

The candidate must have 240 unique `(seed, window)` keys, match all V9 deployment keys, contain valid states, and document that it was computed on the **same 5,000-flow seed orderings**. A matching key schema alone is necessary but not sufficient; the file provenance must establish the same stream data. This cell lists candidates and performs mechanical checks, but deliberately **does not automatically approve provenance**.


In [ ]:
expected=monitor[['seed','window']].drop_duplicates()
checked=[]
for path,sc,wc,tc in valid:
    try:
        df=pd.read_csv(path) if path.suffix.lower()=='.csv' else pd.read_parquet(path)
        t=df[[sc,wc,tc]].rename(columns={sc:'seed',wc:'window',tc:'state'}).copy()
        t['seed']=pd.to_numeric(t.seed,errors='coerce')
        t['window']=pd.to_numeric(t.window,errors='coerce')
        t['state']=t.state.astype(str).str.lower().str.strip()
        valid_states=set(t.state.dropna())<= {'normal','elevated','severe'}
        unique=not t.duplicated(['seed','window']).any()
        aligned=(len(t)==240 and unique and valid_states and
                 set(map(tuple,t[['seed','window']].to_numpy()))==
                 set(map(tuple,expected[['seed','window']].to_numpy())))
        checked.append(dict(path=str(path),rows=len(t),unique_keys=unique,
                            valid_states=valid_states,key_alignment=aligned,
                            provenance_verified=False))
    except Exception as exc:checked.append(dict(path=str(path),error=str(exc)[:180],key_alignment=False,
                                                 provenance_verified=False))
checks=pd.DataFrame(checked)
checks.to_csv(OUT/'WP2_V10_xgboost_alignment_checks.csv',index=False)
if len(checks):display(checks)
print('Transferred baseline NOT evaluated: provenance verification and counterfactual policy outcomes still required.')


## C6 — Operational comparison and manuscript-safe conclusion

The V9 output includes accepted outcomes for `fixed_normal`, `static_severe`, `no_abstention`, and detector-specific policies. **It does not contain the Elevated-threshold counterfactual for every window.** A transferred or ablated policy may choose Elevated on windows where V9 chose Normal or Severe; these outcomes cannot be recovered exactly from aggregate V9 counts.

To finish a *full* V10 comparative study, a follow-up evaluation must run neural inference (no retraining), save per-window outcomes for **all three thresholds** for both detectors, and join those outcomes with a provenance-verified transferred-XGBoost state file. Until then, this notebook provides valid V9 comparisons, exploratory uncertainty, source-only state ablations, and a baseline-readiness audit.


In [ ]:
report={
 'version':'WP2_V10',
 'v8_historical_regression':'previously passed; not rerun',
 'v9_policy_results_loaded':len(results),
 'v9_monitor_windows_loaded':len(monitor),
 'v9_paired_comparisons':'complete',
 'exploratory_block_bootstrap':'complete',
 'source_only_monitor_family_ablation':'complete; monitoring states only',
 'source_only_full_family_regression':'PASS',
 'xgboost_state_schema_candidates':len(valid),
 'xgboost_transferred_policy_outcomes':'NOT EVALUATED',
 'reason':'Require verified stream provenance and per-window all-threshold counterfactuals',
 'deployment_labels_used_for_ablation_calibration':False,
 'claims_to_avoid':['detector-specific beats transferred XGBoost',
                    'ablation improves operational risk',
                    '1% risk is guaranteed under shift']
}
with open(OUT/'WP2_V10_manifest.json','w') as f:json.dump(report,f,indent=2)
print(json.dumps(report,indent=2))
print('Saved V10 artifacts:')
for f in sorted(OUT.glob('WP2_V10_*')):print(f.name)
